# Temporal BioClinicalBERT + LSTM — Multi-label ICD Code Classification

**Our Model**: BioClinicalBERT (note encoder) + LSTM (temporal reasoning)

| Section | Description |
|---------|-------------|
| 0 | Environment Setup |
| 1 | data_loader_temporal |
| 2 | model (Temporal) |
| 3 | evaluate |
| 4 | Hyperparameter Configuration |
| 5 | Build DataLoaders |
| 6 | Build Model & Optimizer |
| 7 | Training Loop |
| 8 | Test Evaluation |
| 9 | Save Results |

## 0. Environment Setup

In [1]:
# Install required packages
!pip install transformers scikit-learn torch -q

In [3]:
# Mount Google Drive (if MIMIC data is stored on Drive)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
MIMIC_DIR = '/content/drive/MyDrive/dataset'  # Update to your actual path
os.listdir(MIMIC_DIR)

['radiology.csv',
 'discharge.csv',
 'edstays.csv',
 'sample_discharge.csv',
 'sample_radiology.csv',
 'D_ICD_DIAGNOSES.csv',
 'DIAGNOSES_ICD.csv',
 'diagnosis.csv']

## 1. data_loader_temporal

In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from pathlib import Path
from collections import Counter


MIMIC_PATH    = Path(MIMIC_DIR)
MODEL_NAME    = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LENGTH    = 512
MAX_NOTES     = 5       # Maximum number of notes per admission (pad/truncate)
TOP_K_CODES   = 50
MIN_CODE_FREQ = 10
BATCH_SIZE    = 8
RANDOM_SEED   = 42


def load_temporal_data(mimic_dir: Path = MIMIC_PATH):
    """Load chronologically ordered note sequences and ICD codes per admission."""
    print('Loading discharge notes...')
    discharge = pd.read_csv(mimic_dir / 'discharge.csv', usecols=['hadm_id', 'charttime', 'text'])
    discharge = discharge.dropna(subset=['hadm_id', 'text'])
    discharge['hadm_id']   = discharge['hadm_id'].astype(int)
    discharge['charttime'] = pd.to_datetime(discharge['charttime'], errors='coerce')
    discharge['note_type'] = 'discharge'
    print(f'    {len(discharge):,} discharge notes loaded')

    print('Loading radiology notes...')
    radiology = pd.read_csv(mimic_dir / 'radiology.csv', usecols=['hadm_id', 'charttime', 'text'])
    radiology = radiology.dropna(subset=['hadm_id', 'text'])
    radiology['hadm_id']   = radiology['hadm_id'].astype(int)
    radiology['charttime'] = pd.to_datetime(radiology['charttime'], errors='coerce')
    radiology['note_type'] = 'radiology'
    print(f'    {len(radiology):,} radiology notes loaded')

    # Combine and sort by admission and time
    all_notes = pd.concat([discharge, radiology], ignore_index=True)
    all_notes = all_notes.sort_values(['hadm_id', 'charttime']).reset_index(drop=True)

    print('Building note sequences per admission...')
    sequences = (
        all_notes.groupby('hadm_id')['text']
        .apply(list).reset_index()
        .rename(columns={'text': 'notes'})
    )
    print(f'    {len(sequences):,} admissions with note sequences')

    print('Loading ICD diagnoses...')
    edstays = pd.read_csv(mimic_dir / 'edstays.csv', usecols=['hadm_id', 'stay_id'])
    edstays = edstays.dropna()
    edstays['hadm_id'] = edstays['hadm_id'].astype(int)
    edstays['stay_id'] = edstays['stay_id'].astype(int)

    diag = pd.read_csv(mimic_dir / 'diagnosis.csv', usecols=['stay_id', 'icd_code', 'icd_version'])
    diag = diag.dropna(subset=['stay_id', 'icd_code'])
    diag['stay_id']  = diag['stay_id'].astype(int)
    diag['icd_code'] = diag['icd_code'].str.strip()
    diag = diag.merge(edstays, on='stay_id', how='inner')
    print(f'    {len(diag):,} diagnosis rows loaded')

    codes_per_adm = (
        diag.groupby('hadm_id')['icd_code']
        .apply(list).reset_index()
        .rename(columns={'icd_code': 'labels'})
    )
    df = sequences.merge(codes_per_adm, on='hadm_id', how='inner')
    print(f'    {len(df):,} admissions after merge')
    return df


def filter_top_codes(df, top_k=TOP_K_CODES, min_freq=MIN_CODE_FREQ):
    """Keep only the top-K most frequent ICD codes."""
    all_codes = [c for codes in df['labels'] for c in codes]
    freq  = Counter(all_codes)
    valid = {c for c, f in freq.items() if f >= min_freq}
    top   = {c for c, _ in freq.most_common(top_k) if c in valid}
    df = df.copy()
    df['labels'] = df['labels'].apply(lambda codes: [c for c in codes if c in top])
    df = df[df['labels'].map(len) > 0].reset_index(drop=True)
    print(f'    {len(top)} ICD codes kept, {len(df):,} admissions remaining')
    return df, sorted(top)


def build_label_binarizer(code_list):
    """Build a multi-label binarizer from a list of ICD codes."""
    mlb = MultiLabelBinarizer(classes=sorted(code_list))
    mlb.fit([code_list])
    return mlb


class TemporalNoteDataset(Dataset):
    """
    Dataset that returns a sequence of tokenized notes per admission.
    Shape per sample: (MAX_NOTES, seq_len). Short sequences are zero-padded.
    """

    def __init__(self, note_sequences, labels_binary, tokenizer,
                 max_length=MAX_LENGTH, max_notes=MAX_NOTES):
        self.sequences  = note_sequences
        self.labels     = labels_binary
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.max_notes  = max_notes

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        notes = self.sequences[idx][:self.max_notes]
        all_input_ids, all_attention_mask = [], []

        for note in notes:
            enc = self.tokenizer(note, max_length=self.max_length,
                                 padding='max_length', truncation=True,
                                 return_tensors='pt')
            all_input_ids.append(enc['input_ids'].squeeze(0))
            all_attention_mask.append(enc['attention_mask'].squeeze(0))

        # Zero-pad to MAX_NOTES
        num_notes = len(notes)
        for _ in range(self.max_notes - num_notes):
            all_input_ids.append(torch.zeros(self.max_length, dtype=torch.long))
            all_attention_mask.append(torch.zeros(self.max_length, dtype=torch.long))

        return {
            'input_ids':      torch.stack(all_input_ids),
            'attention_mask': torch.stack(all_attention_mask),
            'note_mask':      torch.tensor([1] * num_notes + [0] * (self.max_notes - num_notes),
                                           dtype=torch.float32),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float32),
        }


def get_dataloaders(mimic_dir=MIMIC_PATH, top_k=TOP_K_CODES,
                    batch_size=BATCH_SIZE, seed=RANDOM_SEED, num_samples=None):
    """Run the full pipeline and return train/val/test DataLoaders."""
    df = load_temporal_data(mimic_dir)
    df, code_list = filter_top_codes(df, top_k=top_k)
    if num_samples:
        df = df.sample(n=min(num_samples, len(df)), random_state=seed).reset_index(drop=True)
    mlb = build_label_binarizer(code_list)
    y   = mlb.transform(df['labels']).astype(np.float32)
    sequences = df['notes'].tolist()
    idx = list(range(len(sequences)))
    idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=seed)
    idx_val, idx_test   = train_test_split(idx_temp, test_size=0.50, random_state=seed)
    print(f'Loading tokenizer: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    def make_loader(indices, shuffle):
        ds = TemporalNoteDataset([sequences[i] for i in indices], y[indices], tokenizer)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)
    print(f'    Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')
    print(f'    Num classes: {len(code_list)} | Max notes per admission: {MAX_NOTES}')
    return make_loader(idx_train, True), make_loader(idx_val, False), make_loader(idx_test, False), mlb, code_list


print('data_loader_temporal defined successfully')

data_loader_temporal defined successfully


## 2. model.py

In [6]:
import torch
import torch.nn as nn
from transformers import AutoModel


class TemporalClinicalModel(nn.Module):
    """
    Our Model: BioClinicalBERT encoder + LSTM temporal reasoning.

    Key difference from Baseline 1 and 2:
        - Processes multiple notes per admission in chronological order.
        - LSTM captures diagnostic progression across notes over time.
        - Unidirectional LSTM (no future information, consistent with real clinical use).

    Architecture:
        Each note -> BioClinicalBERT [CLS] -> 768-d vector
        Vector sequence -> LSTM -> last hidden state
        -> Linear -> ICD code prediction
    """

    def __init__(self, num_classes: int, lstm_hidden: int = 256,
                 lstm_layers: int = 2, dropout_prob: float = 0.1):
        super().__init__()
        model_name   = 'emilyalsentzer/Bio_ClinicalBERT'
        self.bert    = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_prob)
        bert_hidden  = self.bert.config.hidden_size   # 768

        self.lstm = nn.LSTM(
            input_size=bert_hidden,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout_prob if lstm_layers > 1 else 0,
            bidirectional=False,
        )

        self.classifier = nn.Linear(lstm_hidden, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def encode_notes(self, input_ids, attention_mask):
        """
        Encode a sequence of notes with BioClinicalBERT.

        Args:
            input_ids      : (batch, max_notes, seq_len)
            attention_mask : (batch, max_notes, seq_len)

        Returns:
            note_embeddings: (batch, max_notes, 768)
        """
        B, N, L = input_ids.shape
        # Reshape to (batch * max_notes, seq_len) to pass through BERT at once
        outputs    = self.bert(input_ids=input_ids.view(B * N, L),
                               attention_mask=attention_mask.view(B * N, L))
        cls_embeds = outputs.last_hidden_state[:, 0, :]   # (B*N, 768)
        return self.dropout(cls_embeds).view(B, N, -1)    # (B, N, 768)

    def forward(self, input_ids, attention_mask, note_mask, labels=None):
        """
        Args:
            input_ids      : (batch, max_notes, seq_len)
            attention_mask : (batch, max_notes, seq_len)
            note_mask      : (batch, max_notes) - 1 for real notes, 0 for padding
            labels         : (batch, num_classes) - optional

        Returns:
            loss, logits
        """
        note_embeddings = self.encode_notes(input_ids, attention_mask)  # (B, N, 768)
        lstm_out, _     = self.lstm(note_embeddings)                    # (B, N, lstm_hidden)

        # Use the last real note's hidden state (ignore padding positions)
        note_lengths = note_mask.sum(dim=1).long().clamp(min=1)   # (B,)
        last_hidden  = torch.stack([
            lstm_out[i, note_lengths[i] - 1, :] for i in range(lstm_out.size(0))
        ])   # (B, lstm_hidden)

        logits = self.classifier(last_hidden)   # (B, num_classes)

        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels)

        return loss, logits


def get_model(num_classes: int, device: torch.device) -> TemporalClinicalModel:
    model = TemporalClinicalModel(num_classes=num_classes)
    model.to(device)
    return model


print('model defined successfully')

model defined successfully


## 3. evaluate.py

In [7]:
import torch
import numpy as np
from sklearn.metrics import f1_score


@torch.no_grad()
def evaluate(model, dataloader, device, threshold: float = 0.3) -> dict:
    """
    Args:
        model       : TemporalClinicalModel
        dataloader  : val or test DataLoader
        device      : cuda / cpu
        threshold   : predict positive if sigmoid output >= threshold

    Returns:
        dict with micro_f1, macro_f1, p_at_5, p_at_8
    """
    model.eval()
    all_logits, all_labels = [], []

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        note_mask      = batch['note_mask'].to(device)
        labels         = batch['labels'].cpu().numpy()
        _, logits = model(input_ids, attention_mask, note_mask)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels)

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    probs = 1 / (1 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)

    micro_f1 = f1_score(all_labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_labels, preds, average='macro', zero_division=0)

    return {
        'micro_f1': float(micro_f1),
        'macro_f1': float(macro_f1),
        'p_at_5':   float(precision_at_k(probs, all_labels, k=5)),
        'p_at_8':   float(precision_at_k(probs, all_labels, k=8)),
    }


def precision_at_k(probs: np.ndarray, labels: np.ndarray, k: int) -> float:
    """Precision@k: average fraction of top-k predictions that are true labels."""
    scores = []
    for i in range(probs.shape[0]):
        top_k_idx = np.argsort(probs[i])[::-1][:k]
        scores.append(labels[i][top_k_idx].sum() / k)
    return float(np.mean(scores))


print('evaluate functions defined successfully')

evaluate functions defined successfully


## 4. train.py — Hyperparameter Configuration

In [8]:
CONFIG = {
    'mimic_dir':    MIMIC_DIR,
    'output_dir':   './checkpoints_temporal',
    'top_k_codes':  50,
    'batch_size':   8,
    'max_epochs':   3,
    'lr_bert':      2e-5,   # Learning rate for BioClinicalBERT layers
    'lr_lstm':      1e-3,   # Learning rate for LSTM
    'lr_head':      1e-3,   # Learning rate for classification head
    'warmup_ratio': 0.1,
    'grad_clip':    1.0,
    'threshold':    0.3,    # Sigmoid outputs above this value are predicted positive
    'seed':         42,
    'num_samples':  None,   # Set to an integer (e.g. 2000) for quick testing
}

## 5. Build DataLoaders

In [9]:
train_loader, val_loader, test_loader, mlb, code_list = get_dataloaders(
    mimic_dir=Path(CONFIG['mimic_dir']),
    top_k=CONFIG['top_k_codes'],
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['seed'],
    num_samples=CONFIG['num_samples'],
)
num_classes = len(code_list)
print(f'Number of ICD classes: {num_classes}')

Loading discharge notes...
    331,793 discharge notes loaded
Loading radiology notes...
    1,144,758 radiology notes loaded
Building note sequences per admission...
    374,285 admissions with note sequences
Loading ICD diagnoses...
    421,589 diagnosis rows loaded
    177,676 admissions after merge
    50 ICD codes kept, 101,655 admissions remaining
Loading tokenizer: emilyalsentzer/Bio_ClinicalBERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

    Train: 71,158 | Val: 15,248 | Test: 15,249
    Num classes: 50 | Max notes per admission: 5
Number of ICD classes: 50


## 6. Build Model & Optimizer

In [10]:
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup
from torch.cuda.amp import GradScaler, autocast

torch.manual_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = get_model(num_classes, device)

# Separate learning rates for BERT layers, LSTM, and classification head
optimizer = optim.AdamW([
    {'params': model.bert.parameters(),       'lr': CONFIG['lr_bert']},
    {'params': model.lstm.parameters(),       'lr': CONFIG['lr_lstm']},
    {'params': model.classifier.parameters(), 'lr': CONFIG['lr_head']},
], weight_decay=1e-2)

total_steps  = len(train_loader) * CONFIG['max_epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = GradScaler()
print('Model and optimizer ready')

Device: cuda


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and optimizer ready


/tmp/ipykernel_6739/3750418563.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler       = GradScaler()


## 7. Training Loop

In [ ]:
import time
from pathlib import Path

out_dir = Path(CONFIG['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

best_val_f1 = 0.0
history     = []

print(f"Training for {CONFIG['max_epochs']} epochs | {num_classes} ICD classes\n")

for epoch in range(1, CONFIG['max_epochs'] + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        note_mask      = batch['note_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with autocast():
            loss, _ = model(input_ids, attention_mask, note_mask, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item()
        if step % 50 == 0:
            print(f'  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss    = epoch_loss / len(train_loader)
    val_metrics = evaluate(model, val_loader, device, threshold=CONFIG['threshold'])
    elapsed     = time.time() - t0

    print(f"\nEpoch {epoch}/{CONFIG['max_epochs']} | Loss: {avg_loss:.4f} | "
          f"Val Micro-F1: {val_metrics['micro_f1']:.4f} | "
          f"Val Macro-F1: {val_metrics['macro_f1']:.4f} | Time: {elapsed:.1f}s\n")

    history.append({'epoch': epoch, 'loss': avg_loss, **val_metrics})

    # Save checkpoint every epoch
    torch.save(model.state_dict(), out_dir / 'best_model.pt')
    if val_metrics['micro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['micro_f1']
        print(f'  Best model updated (val micro-F1: {best_val_f1:.4f})\n')

print('Training complete')

Training for 3 epochs | 50 ICD classes



/tmp/ipykernel_6739/2384821672.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1 | Step 50/8895 | Loss: 0.6799
  Epoch 1 | Step 100/8895 | Loss: 0.5046
  Epoch 1 | Step 150/8895 | Loss: 0.1809
  Epoch 1 | Step 200/8895 | Loss: 0.1312
  Epoch 1 | Step 250/8895 | Loss: 0.1400
  Epoch 1 | Step 300/8895 | Loss: 0.0965
  Epoch 1 | Step 350/8895 | Loss: 0.1195
  Epoch 1 | Step 400/8895 | Loss: 0.1319
  Epoch 1 | Step 450/8895 | Loss: 0.1228
  Epoch 1 | Step 500/8895 | Loss: 0.1423
  Epoch 1 | Step 550/8895 | Loss: 0.1029
  Epoch 1 | Step 600/8895 | Loss: 0.1142
  Epoch 1 | Step 650/8895 | Loss: 0.1645
  Epoch 1 | Step 700/8895 | Loss: 0.1134
  Epoch 1 | Step 750/8895 | Loss: 0.1209
  Epoch 1 | Step 800/8895 | Loss: 0.1132
  Epoch 1 | Step 850/8895 | Loss: 0.1190
  Epoch 1 | Step 900/8895 | Loss: 0.1291
  Epoch 1 | Step 950/8895 | Loss: 0.1444
  Epoch 1 | Step 1000/8895 | Loss: 0.1134
  Epoch 1 | Step 1050/8895 | Loss: 0.1393
  Epoch 1 | Step 1100/8895 | Loss: 0.1177
  Epoch 1 | Step 1150/8895 | Loss: 0.1167
  Epoch 1 | Step 1200/8895 | Loss: 0.1149
  Epoch 1 | 

/tmp/ipykernel_6739/2384821672.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 2 | Step 50/8895 | Loss: 0.1018
  Epoch 2 | Step 100/8895 | Loss: 0.0854
  Epoch 2 | Step 150/8895 | Loss: 0.0687
  Epoch 2 | Step 200/8895 | Loss: 0.0670
  Epoch 2 | Step 250/8895 | Loss: 0.0964
  Epoch 2 | Step 300/8895 | Loss: 0.1156
  Epoch 2 | Step 350/8895 | Loss: 0.0741


## 8. Test Evaluation

In [ ]:
print('Evaluating on test set...')
model.load_state_dict(torch.load(out_dir / 'best_model.pt'))
test_metrics = evaluate(model, test_loader, device, threshold=CONFIG['threshold'])

print('\n=== TEST RESULTS (Our Model - Temporal BioClinicalBERT + LSTM) ===')
print(f"  Micro F1   : {test_metrics['micro_f1']:.4f}")
print(f"  Macro F1   : {test_metrics['macro_f1']:.4f}")
print(f"  Precision@5: {test_metrics['p_at_5']:.4f}")
print(f"  Precision@8: {test_metrics['p_at_8']:.4f}")
print('===================================================================')

## 9. Save Results

In [ ]:
import json

results = {'config': CONFIG, 'test_metrics': test_metrics, 'history': history}
with open(out_dir / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {out_dir / 'results.json'}")
print(json.dumps(test_metrics, indent=2))